[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_09/SFM_ch9_seminar/SFM_ch9_seminar.ipynb)

# Statistics of Financial Markets — Seminar 9: ARCH and GARCH models

*Seminar notebook. Daniel Traian Pele, Bucharest University of Economic Studies.*

- This seminar comes before Lecture 9: the slides "What You Need for Today" give every definition used here.
- Part A: computations on paper, checked in code. Part B: real data with inference and an interpretation question. Part C: an open question and an AI answer to audit.
- **[Solved]**: complete code and output, a model to follow. **[Proposed]**: write your own code in the empty cell.

> Quantlet `SFM_ch9_seminar`: student version of the Seminar 9 notebook. Solved exercises (A1, A3, A5, B1, B3, B5) are shown with code and output; the proposed exercises (A2, A4, A6, B2, B4, B6, C1, C2) are not solved here (an empty code cell where they need code).

> The solutions are discussed in the seminar.

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_9'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the arch package (maximum-likelihood estimation of GARCH models) is not part of Google Colab
!pip install -q arch

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

- Transparent background, legend below the plot, course palette.

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

- Daily market data from EODHD, saved in `data/market` of the SFM repository (read locally, or from GitHub in Colab); EUR/RON is the official BNR reference rate.
- Returns in %: $r_t = 100(\ln P_t - \ln P_{t-1})$; each series on its own calendar.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Seminar functions

- Returns; `fit` (maximum likelihood with `arch`), persistence, half-life, a summary of a fit; the news impact curve; the sign-bias test; Ljung–Box; EWMA; out-of-sample forecasts; QLIKE and the Diebold–Mariano test.
- Helpers for Part A: GARCH(1,1) algebra, multi-step forecasts and VaR, the ARCH(1) likelihood step by step, GJR and EGARCH news impact.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from arch import arch_model
NAME = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET', 'btc': 'Bitcoin', 'tlv': 'Banca Transilvania', 'snp': 'OMV Petrom'}
START = {'sp500': '2000-01-01', 'dax': '2000-01-01', 'bet': '2000-01-01', 'btc': None, 'tlv': '2010-01-01', 'snp': '2010-01-01'}
ASSETS = ['sp500', 'dax', 'bet', 'btc', 'tlv', 'snp']
BAD_DAYS = {'tlv': ['2016-05-30', '2016-05-31']}
COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F', 'tlv': '#E67E22', 'snp': '#2E7D32'}
EPISODES = {'2008': ('2008-09-01', '2009-03-31'), '2020': ('2020-02-15', '2020-05-31')}
OOS_START = '2015-01-01'
REFIT = 250
LAMBDA = 0.94
ALPHA_VAR = 0.01
SEED = 2026
TERM_DATES = ['2017-11-03', '2020-03-16']


def returns(k, start=None, end=None):
    """Daily log returns in % on the series' own calendar; known data errors removed."""
    r = log_returns(k, start or START.get(k)) if end is None else log_returns(k, start or START.get(k), end)
    if k in BAD_DAYS:
        r = r.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return r


def fit(r, vol='GARCH', dist='t', o=0, p=1, q=1, last_obs=None, cov_type='robust'):
    """ML estimation with the arch package: constant mean; GARCH (o = 0), GJR-GARCH (o = 1), EGARCH, ARCH(p)."""
    if vol == 'ARCH':
        am = arch_model(r, mean='Constant', vol='ARCH', p=p, dist=dist)
    elif vol == 'EGARCH':
        am = arch_model(r, mean='Constant', vol='EGARCH', p=1, o=1, q=1, dist=dist)
    else:
        am = arch_model(r, mean='Constant', vol='GARCH', p=p, o=o, q=q, dist=dist)
    return am.fit(disp='off', last_obs=last_obs, cov_type=cov_type, options={'maxiter': 2000})


def persistence(res):
    """alpha + beta (GARCH), alpha + beta + gamma/2 (GJR with symmetric innovations), beta (EGARCH)."""
    p = res.params
    if 'gamma[1]' in p and 'alpha[1]' in p and res.model.volatility.__class__.__name__ == 'EGARCH':
        return float(p['beta[1]'])
    return float(p['alpha[1]'] + p['beta[1]'] + 0.5 * p.get('gamma[1]', 0.0))


def half_life(pers):
    """Days after which half of a variance shock has disappeared: ln 0.5 / ln(persistence)."""
    return float(np.log(0.5) / np.log(pers)) if 0 < pers < 1 else float('inf')


def summary(res, r):
    """Parameters, robust standard errors, persistence, half-life and long-run annualised volatility."""
    p, se = res.params, res.std_err
    pers = persistence(res)
    ppy = periods_per_year(r)
    out = {'n': int(res.nobs), 'first': r.index[0].date().isoformat(), 'ppy': float(ppy),
           'params': {k: float(v) for k, v in p.items()}, 'se': {k: float(v) for k, v in se.items()},
           'loglik': float(res.loglikelihood), 'aic': float(res.aic), 'bic': float(res.bic),
           'k': int(len(p)), 'pers': pers, 'hl': half_life(pers),
           'vol_sample': float(r.std() * np.sqrt(ppy))}
    if 'omega' in p and res.model.volatility.__class__.__name__ != 'EGARCH':
        uv = p['omega'] / (1 - pers)
        out['uv'] = float(uv)
        out['vol_lr'] = float(np.sqrt(uv * ppy))
    return out


def news_impact(res, eps, s2bar):
    """sigma_t^2 as a function of the shock eps_{t-1}, with sigma_{t-1}^2 fixed at s2bar (Engle and Ng, 1993)."""
    p = res.params
    if res.model.volatility.__class__.__name__ == 'EGARCH':
        z = eps / np.sqrt(s2bar)
        return np.exp(p['omega'] + p['alpha[1]'] * (np.abs(z) - np.sqrt(2 / np.pi)) + p['gamma[1]'] * z
                      + p['beta[1]'] * np.log(s2bar))
    return p['omega'] + (p['alpha[1]'] + p.get('gamma[1]', 0.0) * (eps < 0)) * eps ** 2 + p['beta[1]'] * s2bar


def sign_bias_test(z, eps):
    """Engle-Ng (1993) sign-bias tests: z_t^2 on a constant, S-_{t-1}, S-_{t-1} eps_{t-1}, S+_{t-1} eps_{t-1};
    t statistics and the joint test T R^2 ~ chi2(3)."""
    d = pd.concat([pd.Series(np.asarray(z)), pd.Series(np.asarray(eps))], axis=1, keys=['z', 'e']).dropna()
    zz, e = d['z'].values, d['e'].values
    y = zz[1:] ** 2
    el = e[:-1]
    sneg = (el < 0).astype(float)
    X = np.column_stack([np.ones_like(el), sneg, sneg * el, (1 - sneg) * el])
    ols = sm.OLS(y, X).fit()
    joint = len(y) * ols.rsquared
    return {'sign_t': float(ols.tvalues[1]), 'neg_t': float(ols.tvalues[2]), 'pos_t': float(ols.tvalues[3]),
            'joint': float(joint), 'joint_p': float(stats.chi2.sf(joint, 3))}


def ljung_box(x, lags=10):
    """Ljung-Box Q(lags) and its p-value."""
    lb = acorr_ljungbox(pd.Series(np.asarray(x)).dropna(), lags=[lags])
    return float(lb['lb_stat'].iloc[0]), float(lb['lb_pvalue'].iloc[0])


def ewma_variance(r, lam=LAMBDA, init=250):
    """EWMA variance forecast for day t made at t-1 (Chapter 8): h_t = lam h_{t-1} + (1 - lam) r_{t-1}^2."""
    x = np.asarray(r, float)
    h = np.empty(len(x))
    h[0] = np.mean(x[:init] ** 2)
    for t in range(1, len(x)):
        h[t] = lam * h[t - 1] + (1 - lam) * x[t - 1] ** 2
    return pd.Series(h, index=r.index)


def oos_forecasts(k, oos_start=OOS_START, refit=REFIT):
    """One-day-ahead variance forecasts on the out-of-sample period: GARCH-t and GJR-t re-estimated every `refit`
    observations on an expanding window (forecast for day t uses data up to t-1), and EWMA; with the GARCH-t and
    EWMA-Normal VaR 1%."""
    r = returns(k)
    i0 = r.index.searchsorted(pd.Timestamp(oos_start))
    cols = {'garch': [], 'gjr': [], 'var_garch': []}
    idx = []
    for s in range(i0, len(r), refit):
        e = min(s + refit, len(r))
        fits = {}
        for lab, o in [('garch', 0), ('gjr', 1)]:
            am = arch_model(r.iloc[:e], mean='Constant', vol='GARCH', p=1, o=o, q=1, dist='t')
            res = am.fit(disp='off', last_obs=r.index[s], options={'maxiter': 2000})
            fixed = am.fix(res.params)
            cols[lab].append(fixed.conditional_volatility.iloc[s:e] ** 2)
            fits[lab] = res.params
        p = fits['garch']
        q = stats.t.ppf(ALPHA_VAR, p['nu']) * np.sqrt((p['nu'] - 2) / p['nu'])
        cols['var_garch'].append(-(p['mu'] + np.sqrt(cols['garch'][-1]) * q))
        idx.append(r.index[s:e])
    df = pd.DataFrame({c: pd.concat(v) for c, v in cols.items()})
    df['ewma'] = ewma_variance(r).loc[df.index]
    df['var_ewma'] = -np.sqrt(df['ewma']) * stats.norm.ppf(ALPHA_VAR)
    df['r'] = r.loc[df.index]
    return df


def qlike(proxy, h):
    """QLIKE loss (Patton, 2011): proxy / h - ln(proxy / h) - 1 is written here as proxy / h + ln h, which differs
    only by a term that does not depend on the forecast and allows a zero proxy."""
    return proxy / h + np.log(h)


def dm_test(la, lb, lags=5):
    """Diebold-Mariano: d = L_a - L_b; HAC (Newey-West) t statistic of the mean of d (negative: model a better)."""
    d = np.asarray(la) - np.asarray(lb)
    ols = sm.OLS(d, np.ones_like(d)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    t = float(ols.tvalues[0])
    return {'mean': float(d.mean()), 't': t, 'p': float(2 * stats.norm.sf(abs(t)))}


def garch_algebra(omega, alpha, beta, s2_t, r_t, mu=0.0, ppy=252):
    """GARCH(1,1) with daily returns in %: persistence, unconditional variance, annualised long-run volatility,
    half-life, and the one-step update sigma^2_{t+1} = omega + alpha (r_t - mu)^2 + beta sigma^2_t."""
    pers = alpha + beta
    uv = omega / (1 - pers)
    return {'pers': pers, 'uv': uv, 'vol_lr': float(np.sqrt(uv * ppy)), 'hl': half_life(pers),
            's2_next': omega + alpha * (r_t - mu) ** 2 + beta * s2_t}


def garch_forecasts(omega, alpha, beta, s2_next, hs=(1, 5, 10, 22), H=10, alpha_var=0.01, nu=None):
    """sigma^2_{t+h} = s2bar + (alpha + beta)^(h-1) (sigma^2_{t+1} - s2bar); the H-day variance as the sum of the
    daily forecasts; one-day and H-day VaR (Normal, or standardised Student-t with nu degrees of freedom)."""
    pers = alpha + beta
    s2bar = omega / (1 - pers)
    f = {h: s2bar + pers ** (h - 1) * (s2_next - s2bar) for h in hs}
    hH = np.arange(1, H + 1)
    sumH = float(np.sum(s2bar + pers ** (hH - 1) * (s2_next - s2bar)))
    q = stats.norm.ppf(alpha_var) if nu is None else stats.t.ppf(alpha_var, nu) * np.sqrt((nu - 2) / nu)
    return {'s2bar': s2bar, 'f': f, 'sumH': sumH, 'volH': np.sqrt(sumH), 'volH_sqrt': np.sqrt(H * s2_next),
            'volH_lr': np.sqrt(H * s2bar), 'q': float(q), 'var1': float(-q * np.sqrt(s2_next)), 'varH': float(-q * np.sqrt(sumH)),
            'factor': float((1 - pers ** H) / (1 - pers))}


def arch1_likelihood(omega, alpha, x):
    """ARCH(1) conditional log-likelihood, step by step: sigma_t^2 = omega + alpha x_{t-1}^2 and
    l_t = -0.5 [ln(2 pi) + ln sigma_t^2 + x_t^2 / sigma_t^2] for t = 2, ..., n; the kurtosis 3(1 - a^2)/(1 - 3a^2)."""
    x = np.asarray(x, float)
    s2 = omega + alpha * x[:-1] ** 2
    lt = -0.5 * (np.log(2 * np.pi) + np.log(s2) + x[1:] ** 2 / s2)
    kurt = 3 * (1 - alpha ** 2) / (1 - 3 * alpha ** 2) if 3 * alpha ** 2 < 1 else float('inf')
    return {'s2': list(s2), 'lt': list(lt), 'loglik': float(lt.sum()), 'kurt': kurt, 'uv': omega / (1 - alpha)}


def asym_news(omega, alpha, gamma, beta, s2_t, shocks=(-2.0, 2.0), eg=None):
    """GJR: sigma^2_{t+1} = omega + (alpha + gamma 1[eps < 0]) eps^2 + beta sigma^2_t for each shock, persistence
    alpha + beta + gamma/2; EGARCH (eg = (omega, alpha, gamma, beta)): ln sigma^2_{t+1} = omega + alpha (|z| -
    sqrt(2/pi)) + gamma z + beta ln sigma^2_t, with z = eps / sigma_t."""
    out = {'gjr': {e: omega + (alpha + gamma * (e < 0)) * e ** 2 + beta * s2_t for e in shocks},
           'pers': alpha + beta + gamma / 2}
    out['hl'] = half_life(out['pers'])
    out['ratio'] = out['gjr'][shocks[0]] / out['gjr'][shocks[1]]
    if eg:
        w, a, g, b = eg
        out['eg'] = {}
        for e in shocks:
            z = e / np.sqrt(s2_t)
            ln = w + a * (abs(z) - np.sqrt(2 / np.pi)) + g * z + b * np.log(s2_t)
            out['eg'][e] = {'ln': ln, 's2': float(np.exp(ln))}
    return out

# Part A: computations on paper

- Do each computation on paper first; then check it with the code.

## A1 [Solved]: GARCH(1,1) algebra

**Context.** Daily returns in %: $\omega = 0.02$, $\alpha = 0.08$, $\beta = 0.90$, $\mu = 0$, 252 trading days per year.

1. Compute the persistence and the long-run variance.
2. Compute the annualised long-run volatility.
3. Compute the half-life of a variance shock.
4. Today $\sigma_t^2 = 1.5$ and $r_t = -3\%$; compute $\sigma_{t+1}^2$ and $\sigma_{t+1}$.

**Report:** four numbers and one sentence.

In [7]:
# Solution
garch_algebra(0.02, 0.08, 0.90, 1.5, -3.0)

{'pers': 0.98,
 'uv': 0.9999999999999991,
 'vol_lr': 15.874507866387537,
 'hl': 34.309618491520645,
 's2_next': 2.09}

**Interpretation of the result.** A fall of 3% raises the variance by about 40%; half of the excess is gone after about 34 trading days.

## A2 [Proposed]: a more reactive market

**Context.** $\omega = 0.05$, $\alpha = 0.12$, $\beta = 0.85$, $\mu = 0$; today $\sigma_t^2 = 2.0$ and $r_t = +1.5\%$. Model: A1.

1. Compute the persistence, the long-run variance and the annualised long-run volatility.
2. Compute the half-life.
3. Compute $\sigma_{t+1}^2$.
4. Write EWMA with $\lambda = 0.94$ as a GARCH(1,1) and say which of the quantities in 1–2 exist for it.

**Report:** four numbers and two sentences.

In [ ]:
# Your code here


## A3 [Solved]: forecasts and VaR 1%

**Context.** The model of A1, with $\sigma_{t+1}^2 = 2.09$ and Normal innovations.

1. Compute $E_t[\sigma_{t+5}^2]$ and $E_t[\sigma_{t+10}^2]$.
2. Compute the 10-day variance and volatility.
3. Compare with the square-root-of-time rule $\sqrt{10}\,\sigma_{t+1}$ and with $\sqrt{10\,\bar\sigma^2}$.
4. Compute the one-day and the 10-day VaR 1%.

**Report:** six numbers and one sentence.

In [9]:
# Solution
garch_forecasts(0.02, 0.08, 0.90, 2.09)

{'s2bar': 0.9999999999999991,
 'f': {1: 2.09,
  5: 2.0053812943999993,
  10: 1.908785060721863,
  22: 1.7131388354287913},
 'sumH': 19.96953202462869,
 'volH': np.float64(4.468728233471878),
 'volH_sqrt': np.float64(4.571651780264984),
 'volH_lr': np.float64(3.1622776601683777),
 'q': -2.3263478740408408,
 'var1': 3.3631621074375184,
 'varH': 10.395816425603584,
 'factor': 9.146359655622657}

**Interpretation of the result.** The high variance decays slowly: the 10-day risk lies between the two simple rules, closer to the square-root-of-time rule.

## A4 [Proposed]: forecasts with Student-t innovations

**Context.** The model of A2, with $\sigma_{t+1}^2 = 2.02$ and standardised Student-t innovations with $\nu = 5$. Model: A3.

1. Compute $E_t[\sigma_{t+10}^2]$ and the 10-day variance.
2. Compute the quantile $q_{0.01}$ of the standardised t.
3. Compute the one-day and the 10-day VaR 1% with t and with Normal innovations.

**Report:** six numbers and one sentence.

In [ ]:
# Your code here


## A5 [Solved]: the ARCH(1) likelihood, step by step

**Context.** ARCH(1) with $\omega = 0.5$, $\alpha = 0.5$, $\mu = 0$, Normal $z_t$; observed $r_1, \dots, r_4 = 0.5, -1.2, 2.0, -0.3$.

1. Compute $\sigma_2^2$, $\sigma_3^2$ and $\sigma_4^2$.
2. Compute $\ell_2$, $\ell_3$, $\ell_4$ and the conditional log-likelihood.
3. Compute the unconditional variance and the kurtosis of this ARCH(1).

**Report:** seven numbers and one sentence.

In [11]:
# Solution
arch1_likelihood(0.5, 0.5, [0.5, -1.2, 2.0, -0.3])

{'s2': [np.float64(0.625), np.float64(1.22), np.float64(2.5)],
 'lt': [np.float64(-1.8359367185818047),
  np.float64(-2.657708224872337),
  np.float64(-1.3950838991417502)],
 'loglik': -5.888728842595892,
 'kurt': 9.0,
 'uv': 1.0}

**Interpretation of the result.** Day 3 (a return of 2 after a calm day) contributes least to the likelihood: a large return when the conditional variance is small is unlikely under the model.

## A6 [Proposed]: good and bad news

**Context.** GJR-GARCH: $\omega = 0.02$, $\alpha = 0.03$, $\gamma = 0.12$, $\beta = 0.90$; EGARCH: $\omega = 0$, $\alpha = 0.12$, $\gamma = -0.08$, $\beta = 0.98$; in both $\sigma_t^2 = 1$, $\mu = 0$. Model: A1.

1. Compute the GJR $\sigma_{t+1}^2$ after $\varepsilon_t = -2$ and after $\varepsilon_t = +2$, and their ratio.
2. Compute the GJR persistence and half-life.
3. Compute the EGARCH $\sigma_{t+1}^2$ for the same two shocks.

**Report:** six numbers and one sentence.

In [ ]:
# Your code here


# Part B: real data, inference and interpretation

## B1 [Solved]: GARCH(1,1)-t for the DAX

**Question.** How persistent is DAX volatility, and how far is the DAX from its long-run volatility today?

1. Estimate GARCH(1,1) with Student-t innovations and report the parameters with robust standard errors.
2. Compute the persistence, the half-life and the annualised long-run volatility, and compare it with the sample volatility.
3. Report the peaks of the annualised conditional volatility in 2008 and in 2020, and its value on the last day.
4. Draw the returns since 2018 with $\pm 2\hat\sigma_t$ bands.
5. Interpretation: should a risk manager use the long-run volatility or the current one for tomorrow?

**Report:** a table of five parameters, five numbers, the chart and two sentences.

In [13]:
# Solution
def b1_estimate(k='dax', save=True):
    """GARCH(1,1)-t: estimates with robust standard errors, persistence, half-life, long-run and sample volatility,
    the volatility peaks of 2008 and 2020; chart of the returns with +/- 2 conditional standard deviations."""
    r = returns(k)
    res = fit(r, dist='t')
    out = summary(res, r)
    ann = np.sqrt(periods_per_year(r))
    v = res.conditional_volatility
    for e, (a, b) in {'2008': ('2008-09-01', '2009-03-31'), '2020': ('2020-02-15', '2020-05-31')}.items():
        w = v.loc[a:b] * ann
        if len(w):
            out[f'peak{e}'] = float(w.max())
            out[f'date{e}'] = w.idxmax().date().isoformat()
    out['last_vol'] = float(v.iloc[-1] * ann)
    out['share_out'] = float((np.abs(r - res.params['mu']) > 2 * v).mean())
    if save:
        a = '2018-01-01'
        fig, ax = plt.subplots(figsize=(10, 3.8))
        ax.plot(r.loc[a:].index, r.loc[a:], color=COLORS[k], lw=0.6, label=f'{NAME[k]}: daily log return (%)')
        ax.plot(v.loc[a:].index, 2 * v.loc[a:], color=st.IDAred, lw=1.1, label='+/- 2 conditional standard deviations, GARCH(1,1)-t')
        ax.plot(v.loc[a:].index, -2 * v.loc[a:], color=st.IDAred, lw=1.1)
        ax.set_ylabel('%')
        st.legend_outside_bottom(ax, ncol=2, y=-0.16)
        st.check_no_grey(fig)
        st.save_fig('ch9_sem_b1')
    return out


b1 = b1_estimate('dax', save=False)
print(pd.DataFrame({'estimate': b1['params'], 'robust SE': b1['se']}).round(4))
{c: b1[c] for c in ['pers', 'hl', 'vol_lr', 'vol_sample', 'peak2008', 'peak2020', 'last_vol', 'share_out']}

          estimate  robust SE
mu          0.0805     0.0118
omega       0.0206     0.0049
alpha[1]    0.0971     0.0113
beta[1]     0.8951     0.0118
nu          7.0790     0.6172


{'pers': 0.9921955575727192,
 'hl': 88.46741343836126,
 'vol_lr': 25.907205171890293,
 'vol_sample': 22.438464466152517,
 'peak2008': 80.39816348957594,
 'peak2020': 81.56923776497017,
 'last_vol': 12.969804893204962,
 'share_out': 0.05157677571470675}

**Interpretation of the result.** For tomorrow the current conditional volatility matters; the long-run level matters only for long horizons, and it is imprecise because $1 - \alpha - \beta$ is small.

## B2 [Proposed]: BET, Banca Transilvania and Bitcoin

**Question.** Which of the BET, Banca Transilvania and Bitcoin has the most persistent volatility? Model: B1.

1. Estimate GARCH(1,1)-t for each series and report $\alpha$, $\beta$ and $\nu$.
2. Compute the persistence, the half-life and the long-run volatility where they exist.
3. Compare the long-run with the sample volatility (annualise Bitcoin with 365 days).
4. Interpretation: what does an estimate $\hat\alpha + \hat\beta = 1$ tell a risk manager?

**Report:** one table and two sentences.

In [ ]:
# Your code here


## B3 [Solved]: the leverage effect in the S&P 500

**Question.** Do falls raise the volatility of the S&P 500 more than rises of the same size?

1. Estimate GARCH(1,1)-t and GJR-GARCH(1,1)-t; report $\gamma$ with its robust $t$ statistic.
2. Compute the LR statistic of GJR against GARCH, its p-value and the two BIC values.
3. Run the sign-bias test on the standardised residuals of GARCH-t.
4. Draw both news impact curves and compute the GJR variance after shocks of −2% and +2%.
5. Interpretation: how does the leverage effect change the VaR on the day after a fall?

**Report:** four statistics with p-values, two variances, the chart and two sentences.

In [15]:
# Solution
def b3_asymmetry(k='sp500', save=True):
    """GARCH-t against GJR-t: gamma with its robust t statistic, the likelihood-ratio test, BIC, the sign-bias test on
    the GARCH-t residuals, and the GJR variance after shocks of -2% and +2%; chart of the two news impact curves."""
    r = returns(k)
    rg, rj = fit(r, dist='t'), fit(r, o=1, dist='t')
    lr = 2 * (rj.loglikelihood - rg.loglikelihood)
    s2bar = float(r.var())
    nic = {e: float(news_impact(rj, np.array([e]), s2bar)[0]) for e in (-2.0, 2.0)}
    out = {'gamma': float(rj.params['gamma[1]']), 't_gamma': float(rj.tvalues['gamma[1]']),
           'alpha_gjr': float(rj.params['alpha[1]']), 'beta_gjr': float(rj.params['beta[1]']),
           'pers_gjr': persistence(rj), 'pers_garch': persistence(rg), 'lr': float(lr), 'p_lr': float(stats.chi2.sf(lr, 1)),
           'bic_garch': float(rg.bic), 'bic_gjr': float(rj.bic), 'sb': sign_bias_test(rg.std_resid, rg.resid),
           'nic_neg': nic[-2.0], 'nic_pos': nic[2.0], 'ratio': nic[-2.0] / nic[2.0], 's2bar': s2bar}
    if save:
        g = np.linspace(-4, 4, 201)
        fig, ax = plt.subplots(figsize=(10, 3.8))
        ax.plot(g, news_impact(rg, g, s2bar), color=st.MainBlue, lw=1.6, label='GARCH(1,1)-t')
        ax.plot(g, news_impact(rj, g, s2bar), color=st.IDAred, lw=1.6, label='GJR-GARCH(1,1)-t')
        ax.set_xlabel('shock yesterday, eps(t-1) (%)')
        ax.set_ylabel('variance today (%^2)')
        st.legend_outside_bottom(ax, ncol=2, y=-0.2)
        st.check_no_grey(fig)
        st.save_fig('ch9_sem_b3')
    return out


b3 = b3_asymmetry('sp500', save=False)
b3

{'gamma': 0.20518725729389392,
 't_gamma': 10.027735545530913,
 'alpha_gjr': 0.0,
 'beta_gjr': 0.8808581501089234,
 'pers_gjr': 0.9834517787558704,
 'pers_garch': 0.9947120452816396,
 'lr': 235.65251563321726,
 'p_lr': 3.4888848674232973e-53,
 'bic_garch': 18169.006314276925,
 'bic_gjr': 17942.165748821248,
 'sb': {'sign_t': 3.5644971855836114,
  'neg_t': 1.1291832329231297,
  'pos_t': -2.612821504737135,
  'joint': 40.66475977251353,
  'joint_p': 7.70229184039106e-09},
 'nic_neg': 2.135562479279996,
 'nic_pos': 1.3148134501044202,
 'ratio': 1.6242323039138316,
 's2bar': 1.4714407863451462}

**Interpretation of the result.** Only falls raise the variance ($\hat\alpha = 0$ in GJR); after a fall the VaR is about $\sqrt{1.6}$ times the VaR after an equal rise; a symmetric GARCH understates risk after bad days.

## B4 [Proposed]: asymmetry on the BVB and in Bitcoin

**Question.** Is there a leverage effect in the BET, in Banca Transilvania and in Bitcoin? Model: B3.

1. Estimate GJR-GARCH(1,1)-t and EGARCH(1,1)-t; report both $\gamma$ with robust $t$ statistics.
2. Compute the LR test of GJR against GARCH and the change in BIC.
3. Compute the GJR ratio of the variances after −2% and +2%.
4. Interpretation: would you use an asymmetric model for each of the three? Why?

**Report:** one table and two sentences.

In [ ]:
# Your code here


## B5 [Solved]: GARCH against EWMA for the S&P 500

**Question.** Does GARCH(1,1)-t forecast tomorrow's variance of the S&P 500 better than EWMA, and is its VaR 1% exceeded on 1% of days?

1. Re-estimate GARCH(1,1)-t every 250 days on all data up to that day and compute the one-day forecasts; compute EWMA with $\lambda = 0.94$.
2. Compute the mean QLIKE of both models and the Diebold–Mariano $t$ statistic.
3. Count the exceedances of the GARCH-t VaR 1% and of the EWMA-Normal VaR 1% and compare with the expected number.
4. Draw the cumulative difference of the losses.
5. Interpretation: which model would you give to a risk manager, and what would you still fix?

**Report:** four numbers, two counts, the chart and two sentences.

In [17]:
# Solution
def b5_oos(k='sp500', save=True):
    """Out-of-sample one-day forecasts since 2015 (GARCH-t re-estimated every 250 days, EWMA lambda = 0.94): mean
    QLIKE, the Diebold-Mariano test, the VaR 1% exceedances; chart of the cumulative QLIKE difference."""
    df = oos_forecasts(k)
    px = df['r'] ** 2
    lg, le = qlike(px, df['garch']), qlike(px, df['ewma'])
    eg, ee = df['r'] < -df['var_garch'], df['r'] < -df['var_ewma']
    out = {'n': int(len(df)), 'first': df.index[0].date().isoformat(), 'q_garch': float(lg.mean()), 'q_ewma': float(le.mean()),
           'dm': dm_test(lg, le), 'nexc_garch': int(eg.sum()), 'nexc_ewma': int(ee.sum()),
           'exc_garch': float(eg.mean()), 'exc_ewma': float(ee.mean()), 'expected': float(0.01 * len(df)),
           'last_var_garch': float(df['var_garch'].iloc[-1]), 'last_var_ewma': float(df['var_ewma'].iloc[-1])}
    if save:
        d = (le - lg).cumsum()
        fig, ax = plt.subplots(figsize=(10, 3.6))
        ax.plot(d.index, d.values, color=COLORS[k], lw=1.4, label=f'{NAME[k]}: cumulative QLIKE(EWMA) - QLIKE(GARCH-t)')
        ax.axhline(0, color='black', lw=0.8, ls='--')
        ax.set_ylabel('cumulative loss difference')
        st.legend_outside_bottom(ax, ncol=1, y=-0.16)
        st.check_no_grey(fig)
        st.save_fig('ch9_sem_b5')
    return out


b5 = b5_oos('sp500', save=False)
b5

{'n': 2944,
 'first': '2015-01-02',
 'q_garch': 0.7611897391103296,
 'q_ewma': 0.818761373078798,
 'dm': {'mean': -0.057571633968468254,
  't': -3.6888682742772287,
  'p': 0.00022525379812602132},
 'nexc_garch': 49,
 'nexc_ewma': 65,
 'exc_garch': 0.016644021739130436,
 'exc_ewma': 0.022078804347826088,
 'expected': 29.44,
 'last_var_garch': 1.8449625790667268,
 'last_var_ewma': 1.5681890998790926}

**Interpretation of the result.** GARCH-t has the smaller QLIKE (significant by Diebold–Mariano), but both VaR models are exceeded more often than on 1% of the days: add the leverage effect (B3) and test the exceedances formally (Chapter 10).

## B6 [Proposed]: GARCH against EWMA for the DAX and the BET

**Question.** Does the result of B5 hold for the DAX and for the BET? Model: B5.

1. Compute the out-of-sample GARCH(1,1)-t and EWMA forecasts as in B5.
2. Compute the mean QLIKE of both models and the DM test.
3. Count the VaR 1% exceedances of both models.
4. Interpretation: for which index does GARCH help most, and why?

**Report:** one table and two sentences.

In [ ]:
# Your code here


# Part C: open questions and AI critique

## C1 [Proposed]: is Bitcoin volatility becoming equity-like?

**Question.** Has Bitcoin volatility moved closer to the volatility of the S&P 500 since 2021? Models: B1, B3.

1. Estimate GJR-GARCH(1,1)-t for both series in 2015–2019 and in 2021–2026 and report the persistence, $\gamma$ and $\nu$.
2. Compute the correlation of the logarithms of the two GARCH-t volatilities on common days in each period.
3. List the events of 2020–2024 that could explain a change.
4. Interpretation: how would you separate a lasting change from a change caused by one crisis?

**Report:** a table and a plan for a project.

In [ ]:
# Your code here


## C2 [Proposed]: audit an AI answer

**Context.** An AI assistant interpreted a GARCH(1,1)-t fit of the S&P 500 (daily, since 2000). It answered:

- (a) $\alpha + \beta$ is close to 1, so a volatility shock disappears within about a week;
- (b) the long-run variance is $\omega/(1 - \alpha)$;
- (c) a GJR fit gives $\gamma > 0$: good news raises volatility more than bad news;
- (d) the Ljung–Box test of the squared standardised residuals does not reject, so the GARCH-t model is correct;
- (e) with Student-t innovations the one-day VaR 1% is 2.326 times $\sigma_{t+1}$;
- (f) EWMA with $\lambda = 0.94$ is a GARCH(1,1) with $\omega = 0$, $\alpha = 0.06$, $\beta = 0.94$.

1. For each statement, say whether it is correct; if not, give the correct statement and the correct number.

**Report:** six verdicts with one line of justification each.

In [ ]:
# Your code here
